In [1]:
!pip install transformers accelerate datasets scikit-learn pandas

In [2]:
#import needed libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import re

#read in the data
fakeData = pd.read_csv("Fake.csv")
trueData = pd.read_csv("True.csv")

#add labels that are standard for AI models. indicated 0 for fake news and 1 for true news
fakeData['label'] = 0
trueData['label'] = 1

#the dataset includes clear indicators of true value so get rid of that string for a more advanced ai model
def cleanData(article):
    #split the article when it finds the string '(Reuters) -'
    identified = article.split('(Reuters) - ', 1)
    if len(identified) > 1:
        #return the text that comes after (Reuters)
        return identified[1]
    #if it's not in the test, just return the entire article
    return article

#update the true csv file 
trueData['identified'] = trueData['text'].apply(cleanData)
fakeData['identified'] = fakeData['text']

#only extract 500 lines of the datasets
useFake = fakeData.sample(n=250, random_state=42)
useTrue = trueData.sample(n=250, random_state=42)

#combine the datasets, randomize the order, delete the index numbers
combineFT = pd.concat([useFake, useTrue])
randomFT = combineFT.sample(frac=1, random_state=42)
finalData = randomFT.reset_index(drop=True)

#split train and test data
trainData, testData, trainLabel, testLabel = train_test_split(
    finalData['identified'].tolist(), finalData['label'].tolist(), test_size=0.2, random_state=42
)

print(f"Train: {len(trainData)}, Test: {len(testData)}")

Train: 400, Test: 100


In [3]:
#the naive baseline
from sklearn.metrics import classification_report

def baseline(article):
    #standard for baseline is the number of capital letters in the title of the article
    upperCount = sum(1 for c in article if c.isupper())
    length = len(article)

    #if more than 5% of the article is uppercase, predict that it is fake news
    if length > 0 and (upperCount / length) > 0.05:
        return 0
    
    return 1

baselineResult = [baseline(str(t)) for t in testData]

print("Naive Baseline Results")
print(classification_report(testLabel, baselineResult, target_names=['Fake', 'True']))

Naive Baseline Results
              precision    recall  f1-score   support

        Fake       0.67      0.19      0.29        43
        True       0.60      0.93      0.73        57

    accuracy                           0.61       100
   macro avg       0.63      0.56      0.51       100
weighted avg       0.63      0.61      0.54       100



In [4]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset

#setup the distilBERT ai pipeline model
modelName = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(modelName)
model = DistilBertForSequenceClassification.from_pretrained(modelName, num_labels=2)

#change into strings to solve TypeError
trainDataList = [str(x) for x in trainData]
testDataList = [str(x) for x in testData]

#tokenize the data
trainToken = tokenizer(trainDataList, truncation=True, padding=True, max_length=128)
testToken = tokenizer(testDataList, truncation=True, padding=True, max_length=128)

#create an object for dataset pytorch
class newsData(Dataset):
    def __init__(self, tokenized, label):
        self.tokenized = tokenized
        self.label = label

    def __len__(self):
        return len(self.label)

    def __getitem__(self, idx):
        #get calculated numbers from tokenizer and add to the label
        item = {key: torch.tensor(val[idx]) for key, val in self.tokenized.items()}
        item['labels'] = torch.tensor(self.label[idx])
        return item

trainDataset = newsData(trainToken, trainLabel)
testDataset = newsData(testToken, testLabel)

#the actual training process
trainArgs = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    logging_dir='./logs',
)

trainer = Trainer(
    model=model,
    args=trainArgs,
    train_dataset=trainDataset,
    eval_dataset=testDataset
)

trainer.train()


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/tokenizer_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7f5a863bec80>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: b09a98b2-5f0b-4e88-9ea5-2718a059b748)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the pr

Step,Training Loss


TrainOutput(global_step=100, training_loss=0.1976120376586914, metrics={'train_runtime': 4.3955, 'train_samples_per_second': 182.006, 'train_steps_per_second': 22.751, 'total_flos': 26493479731200.0, 'train_loss': 0.1976120376586914, 'epoch': 2.0})

In [5]:
import numpy as np

#get the predicted results
predResults = trainer.predict(testDataset)
aiPred = np.argmax(predResults.predictions, axis=1)

print("\n AI Model Pipeline Results")
print(classification_report(testLabel, aiPred, target_names=['Fake', 'True']))

#counter for the loop
count = 0

#get the comparisons
print("\nBaseline vs AI model pipeline")
for l in range(len(testLabel)):
    if baselineResult[l] != testLabel[l] and aiPred[l] == testLabel[l]:
        print(f"Index: {l}")
        print(f"True or Fake: {testLabel[l]}")
        print(f"Baseline Prediction: {baselineResult[l]}")
        print(f"AI Prediction: {aiPred[l]}")
        print(f"Tested Article: {str(testData[l])[:100]}...")
        print("\n")

        count += 1
        if count >= 3:
            break


 AI Model Pipeline Results
              precision    recall  f1-score   support

        Fake       1.00      0.91      0.95        43
        True       0.93      1.00      0.97        57

    accuracy                           0.96       100
   macro avg       0.97      0.95      0.96       100
weighted avg       0.96      0.96      0.96       100


Baseline vs AI model pipeline
Index: 9
True or Fake: 0
Baseline Prediction: 1
AI Prediction: 0
Tested Article: Make America Conservative again. There really isn t any downside It might be time to rethink the mil...


Index: 11
True or Fake: 0
Baseline Prediction: 1
AI Prediction: 0
Tested Article: As we all live in the nightmarish hellscape that is the Donald Trump presidency is the fact that Con...


Index: 12
True or Fake: 1
Baseline Prediction: 0
AI Prediction: 1
Tested Article: President Donald Trump has picked former U.N. spokesman Richard Grenell as U.S. ambassador to German...


